In [ ]:
!pip install rdkit
!pip install Bio
!pip install cudaq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 MB 6.7 MB/s eta 0:00:00
  Created wheel for cudaq: filename=cudaq-0.12.0-py3-none-any.whl size=7044 sha256=fb876acda60f8d03999e980c7236551dbfc13fc1032fcd99e06645892d348794
  Stored i

In [ ]:
import os, math
import numpy as np
import networkx as nx
from rdkit import Chem, RDConfig
from rdkit.Chem import AllChem, ChemicalFeatures, rdchem, Descriptors, Lipinski, Crippen
from Bio.PDB import PDBParser, NeighborSearch
import cudaq
from cudaq import spin
import itertools
#Dont run more than once
# binding_scores, ligand_smiles_list = [], []


def rdkit_ligand_features(mol, confId=0, families=None):
    fdef = os.path.join(RDConfig.RDDataDir, "BaseFeatures.fdef")
    factory = ChemicalFeatures.BuildFeatureFactory(fdef)
    feats = []
    for f in factory.GetFeaturesForMol(mol, confId=confId):
        fam = f.GetFamily()
        if (families is None) or (fam in families):
            feats.append({
                "id": ("L", len(feats)),
                "type": fam,
                "pos": np.array([f.GetPos().x, f.GetPos().y, f.GetPos().z])
            })
    return feats

def protein_features_from_pdb(pdb_path, site_center=None, site_radius=8.0):
    """Return receptor feature points {id,type,pos} within site sphere."""
    structure = PDBParser(QUIET=True).get_structure("prot", pdb_path)
    atoms = [a for a in structure.get_atoms() if a.element != "H"]

    if site_center is None:
        coords = np.array([a.coord for a in atoms])
        site_center = coords.mean(0)

    feats = []
    for a in atoms:
        p = a.coord
        if np.linalg.norm(p - site_center) > site_radius:
            continue
        res = a.get_parent()
        resname = res.get_resname().strip()
        aname = a.get_name().strip()

        if aname == "O":
            feats.append({"id": ("R", len(feats)), "type": "Acceptor", "pos": p.copy()})


        if aname in ("N","NE","NE2","ND2","NZ"):
            feats.append({"id": ("R", len(feats)), "type": "Donor", "pos": p.copy()})

        if resname in ("LYS","ARG") and aname in ("NZ","CZ","NE","NH1","NH2"):
            feats.append({"id": ("R", len(feats)), "type": "PosIonizable", "pos": p.copy()})
        if resname in ("ASP","GLU") and aname.startswith("O"):
            feats.append({"id": ("R", len(feats)), "type": "NegIonizable", "pos": p.copy()})

        aromatic_res = {"PHE":{"CG","CD1","CD2","CE1","CE2","CZ"},
                        "TYR":{"CG","CD1","CD2","CE1","CE2","CZ"},
                        "TRP":{"CD2","CE2","CE3","CZ2","CZ3","CH2"},
                        "HIS":{"CG","ND1","CD2","CE1","NE2"}}
        if resname in aromatic_res and aname in aromatic_res[resname]:
            pass

    return feats

COMPLEMENT = {
    ("Donor","Acceptor"), ("Acceptor","Donor"),
    ("PosIonizable","NegIonizable"), ("NegIonizable","PosIonizable"),
    ("Aromatic","Aromatic"),
    ("Hydrophobe","Hydrophobe"),
}

def in_contact_window(tL, tR, dist):
    if {tL,tR}=={"Donor","Acceptor"}:      return 1.6 <= dist <= 3.3
    if {tL,tR}=={"PosIonizable","NegIonizable"}: return 2.0 <= dist <= 5.0
    if tL==tR=="Aromatic":                 return 3.5 <= dist <= 6.0
    if tL==tR=="Hydrophobe":               return 3.0 <= dist <= 6.5
    return False

def build_BIG(lig_feats, rec_feats, eps_pair=0.75, max_candidates_per_lig=20):
    nodes = []
    for i, lf in enumerate(lig_feats):
        candidates = []
        for j, rf in enumerate(rec_feats):
            if (lf["type"], rf["type"]) not in COMPLEMENT:
                continue
            d = np.linalg.norm(lf["pos"] - rf["pos"])
            if in_contact_window(lf["type"], rf["type"], d):
                candidates.append((d, i, j))
        candidates.sort()
        for _, i2, j in candidates[:max_candidates_per_lig]:
            nodes.append((i2, j))

    G = nx.Graph()
    G.add_nodes_from(nodes)

    for a in range(len(nodes)):
        i, j = nodes[a]
        xi, yj = lig_feats[i]["pos"], rec_feats[j]["pos"]
        for b in range(a+1, len(nodes)):
            k, m = nodes[b]
            if (i==k) or (j==m):
                continue
            xk, ym = lig_feats[k]["pos"], rec_feats[m]["pos"]
            if abs(np.linalg.norm(xi - xk) - np.linalg.norm(yj - ym)) <= eps_pair:
                G.add_edge((i,j), (k,m))
    return G

def kabsch(P, Q):
    Pc = P.mean(0); Qc = Q.mean(0)
    X = P - Pc; Y = Q - Qc
    H = X.T @ Y
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1,:] *= -1
        R = Vt.T @ U.T
    t = Qc - R @ Pc
    return R, t

def pose_from_clique(clique, lig_feats, rec_feats):
    P = np.array([lig_feats[i]["pos"] for (i,_) in clique])
    Q = np.array([rec_feats[j]["pos"] for (_,j) in clique])
    R, t = kabsch(P, Q)
    rms = np.sqrt(((Q - (P @ R.T + t))**2).sum(axis=1).mean())
    return R, t, rms

# Use the Methods to Create a Binding-Interaction Graph Between Ligand and Protein

In [ ]:
# Obtain SMILES String of ligand of interest from database
smiles = 'CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C4=CC=CC=N4' # Replace every time
lig = Chem.AddHs(Chem.MolFromSmiles(smiles))
AllChem.EmbedMolecule(lig, AllChem.ETKDG())
AllChem.MMFFOptimizeMolecule(lig)
lig_feats = rdkit_ligand_features(lig, confId=0, families=None)

# Obtain AlphaFold multimer result as a single PDB file of atomic coordinates
pdb_path = '/content/unrelaxed_model_1_pred_0.pdb' # Replace

# Identify Location of Binding site of interest (It's 3D coordinate in the AlphaFold-returned PDB)
binding_site_center = np.array([0, 0, 0]) # Replace
binding_site_radius = 8.0 # Defines a sphere with this radius around the binding center and collects pharamcophores in this sphere only
rec_feats = protein_features_from_pdb(pdb_path, site_center=binding_site_center, site_radius=binding_site_radius)

# Build the BIG
G = build_BIG(lig_feats, rec_feats, eps_pair=0.6, max_candidates_per_lig=15)

NameError: name 'Chem' is not defined

In [ ]:

node2idx = {n: k for k, n in enumerate(G.nodes())}
idx2node = {k: n for n, k in node2idx.items()}

nodes = list(idx2node.keys())
qubit_num = len(nodes)


edges = [(node2idx[u], node2idx[v]) for u, v in G.edges()]

non_edges = [
    (u, v)
    for u, v in itertools.combinations(nodes, 2)
    if (u, v) not in edges and (v, u) not in edges
]

weights = []
for k in nodes:
    iL, jR = idx2node[k]
    d = np.linalg.norm(lig_feats[iL]["pos"] - rec_feats[jR]["pos"])
    weights.append(1.0 / d)

penalty = 1.2 * max(weights)            # rule‑of‑thumb ≥ 1.2 × max weight
num_layers = 3#Chosen experimentally


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



ValueError: max() arg is an empty sequence

During handling of the above exception, another exception occurred:

AttributeError: 'ValueError' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

AssertionError
ValueError: max() arg is an empty sequence

During handling of the above exception, another exception occurred:

AttributeError: 'ValueError' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

TypeError: object of type 'NoneType' has no len()

During handling of the above exception, another exception occurred:

AttributeError: 'TypeError' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

AssertionError
ValueError: max() arg is an empty sequence

During handling of the above exception, another exception occurred:

AttributeError: 'ValueError' object has no attribute '_render_traceback_'

Durin

In [ ]:

# # BIG 1 (Example from Nvidia)

# nodes = [0, 1, 2, 3, 4, 5] #Depends on number of ph4s
# qubit_num = len(nodes)
# edges = [[0, 1], [0, 2], [0, 4], [0, 5], [1, 2], [1, 3], [1, 5], [2, 3], [2, 4],
#          [3, 4], [3, 5], [4, 5]] #from checking every node pair against clashes
# non_edges = [
#     [u, v] for u in nodes for v in nodes if u < v and [u, v] not in edges
# ]

# print('Edges: ', edges)
# print('Non-Edges: ', non_edges)

# weights = [0.6686, 0.6686, 0.6686, 0.1453, 0.1453, 0.1453]
# penalty = 6.0
# #Set by user, penalizes edges of the graph with no interactions
# #rule of thumb is P ≳ 1.2 × (max weight)
# num_layers = 3#Chosen experimentally

In [ ]:
# Hamiltonian
def ham_clique(penalty, nodes, weights, non_edges) -> cudaq.SpinOperator:

    spin_ham = 0
    for wt, node in zip(weights, nodes):
        #print(wt,node)
        spin_ham += 0.5 * wt * spin.z(node)
        spin_ham -= 0.5 * wt * spin.i(node)

    for non_edge in non_edges:
        u, v = (non_edge[0], non_edge[1])
        #print(u,v)
        spin_ham += penalty / 4.0 * (spin.z(u) * spin.z(v) - spin.z(u) -
                                     spin.z(v) + spin.i(u) * spin.i(v))

    return spin_ham

In [ ]:
def term_coefficients(ham: cudaq.SpinOperator) -> list[complex]:
    result = []
    for term in ham:
        result.append(term.evaluate_coefficient())
    return result

    # Get corresponding Pauli wordss


def term_words(ham: cudaq.SpinOperator) -> list[str]:
    # Our kernel uses these words to apply exp_pauli to the entire state.

    result = []
    for term in ham:
        result.append(term.get_pauli_word(qubit_num))
    return result


ham = ham_clique(penalty, nodes, weights, non_edges)
print(ham)

coef = term_coefficients(ham)
words = term_words(ham)

print(term_coefficients(ham))
print(term_words(ham))

(0+0i) + (-0.634191+0i) * Z0 + (-0.155004+0i) * I0 + (-0.600184+0i) * Z1 + (-0.18901+0i) * I1 + (-0.629958+0i) * Z2 + (-0.159236+0i) * I2 + (-0.438441+0i) * Z3 + (-0.219221+0i) * I3 + (-0.634894+0i) * Z4 + (-0.1543+0i) * I4 + (-0.602436+0i) * Z5 + (-0.186759+0i) * I5 + (-0.493006+0i) * Z6 + (-0.164656+0i) * I6 + (0.131532+0i) * Z0Z1 + (0.131532+0i) * I0I1 + (0.131532+0i) * Z0Z2 + (0.131532+0i) * I0I2 + (0.131532+0i) * Z0Z3 + (0.131532+0i) * I0I3 + (0.131532+0i) * Z0Z4 + (0.131532+0i) * I0I4 + (0.131532+0i) * Z0Z5 + (0.131532+0i) * I0I5 + (0.131532+0i) * Z0Z6 + (0.131532+0i) * I0I6 + (0.131532+0i) * Z1Z2 + (0.131532+0i) * I1I2 + (0.131532+0i) * Z1Z3 + (0.131532+0i) * I1I3 + (0.131532+0i) * Z1Z4 + (0.131532+0i) * I1I4 + (0.131532+0i) * Z1Z5 + (0.131532+0i) * I1I5 + (0.131532+0i) * Z1Z6 + (0.131532+0i) * I1I6 + (0.131532+0i) * Z2Z3 + (0.131532+0i) * I2I3 + (0.131532+0i) * Z2Z4 + (0.131532+0i) * I2I4 + (0.131532+0i) * Z2Z5 + (0.131532+0i) * I2I5 + (0.131532+0i) * Z2Z6 + (0.131532+0i) * I2I

In [ ]:
@cudaq.kernel
def dc_qaoa(qubit_num:int, num_layers:int, thetas:list[float],\
    coef:list[complex], words:list[cudaq.pauli_word]):

    qubits = cudaq.qvector(qubit_num)

    h(qubits)

    count = 0
    for p in range(num_layers):

        for i in range(len(coef)):
            exp_pauli(thetas[count] * coef[i].real, qubits, words[i])
            count += 1

        for j in range(qubit_num):
            rx(thetas[count], qubits[j])
            count += 1

        #Comment out this for loop for conventional QAOA
        for k in range(qubit_num):
            ry(thetas[count], qubits[k])
            count += 1

In [ ]:
# Specify the optimizer and its initial parameters.
optimizer = cudaq.optimizers.NelderMead()

#Specify random seeds
np.random.seed(13)
cudaq.set_random_seed(13)

# if dc_qaoa used
parameter_count = (2 * qubit_num + len(coef)) * num_layers

# if qaoa used
# parameter_count=(qubit_num+len(coef))*num_layers

print('Total number of parameters: ', parameter_count)
optimizer.initial_parameters = np.random.uniform(-np.pi / 8, np.pi / 8,
                                                 parameter_count)
print("Initial parameters = ", optimizer.initial_parameters)

Total number of parameters:  207
Initial parameters =  [0.21810696323572243, -0.20613464375211488, 0.2546877639814583, 0.3657985647468064, 0.37118004688049144, -0.03656087558321203, 0.08564174998504231, 0.21639801853794682, 0.11122286088634259, 0.1743727097033635, -0.36518146001762486, -0.15829741539542244, -0.3467434780387345, 0.28043500852894776, -0.09986021299050934, 0.14125225086023052, -0.19141728018199775, -0.11970943368650361, -0.3853063093646483, -0.1112643868789806, 0.3527177454825464, -0.22156160012057186, -0.1418496891385843, 0.32811766468303116, -0.367642000671186, -0.34158180583996006, 0.10196745745501312, 0.29359239180502594, -0.3858537615546677, 0.19366130907065582, 0.24570488114056754, -0.3332307385378807, 0.12287973244618389, 0.007274514934614895, -0.015799547372526146, 0.3578070967202224, -0.39268963055535144, -0.19872246354138554, 0.16668715544467982, -0.13777293592446055, -0.17514665212709513, 0.15350249947988204, 0.32872977428061945, -0.20068831419712105, -0.032919

In [ ]:
cost_values = []


def objective(parameters):

    cost = cudaq.observe(dc_qaoa, ham, qubit_num, num_layers, parameters, coef,
                         words).expectation()
    cost_values.append(cost)
    return cost


# Optimize!
optimal_expectation, optimal_parameters = optimizer.optimize(
    dimensions=parameter_count, function=objective)

print('optimal_expectation =', optimal_expectation)
print('optimal_parameters =', optimal_parameters)

optimal_expectation = -0.7674287004159897
optimal_parameters = [1.712175754348245, 1.2843958730857519, 1.7052883478071972, 0.6029580563819796, 0.3631030276823401, 1.1976730626049397, 1.3798763443624158, 0.061768503463085045, 0.040104321007268205, 0.3300668868391453, -0.41391065059605225, 1.3001580962993478, 0.36443263166354434, 0.828833128329163, 0.3580083279861386, 0.28050010423349075, -0.11123786378067382, -0.16336032118999788, -0.25936852545359107, 0.008725427063549916, 0.5622661596062082, 0.018385065199470897, 0.32241567675510485, -0.00318757186427432, -0.3278250393862536, 0.009313986619458026, 0.4915571623676661, 0.07202760653497453, -0.37931015418396774, 0.2734110125178488, 0.2331724492849679, 0.005135556330207856, 0.3161996590130236, -0.0053257603655111, -0.11061174687462753, 0.37680962991687406, -0.30550232020431944, -0.19951637121902166, 0.15973643857192926, 0.018897552910997865, -0.26868529223120585, 0.007202893969526414, 0.1902098771142478, -0.2553148790868189, -0.1386668928

In [ ]:
shots = 200000

counts = cudaq.sample(dc_qaoa,
                      qubit_num,
                      num_layers,
                      optimal_parameters,
                      coef,
                      words,
                      shots_count=shots)
print(counts)

print('The MVWCP is given by the partition: ', counts.most_probable())

{ 0000000:4 0000011:1 0000101:5 0001001:199928 0010000:7 0010001:7 0100000:7 0100001:3 0101000:1 0111000:5 0111001:2 1001001:1 1001011:1 1010001:6 1011000:7 1011101:1 1100000:3 1100001:2 1101000:6 1101001:1 1111001:2 }

The MVWCP is given by the partition:  0001001


In [ ]:
binding_score = -optimal_expectation
ligand_smiles_list.append(smiles)
binding_scores.append(binding_score)
print(binding_scores)
print(set(ligand_smiles_list))

[0.4914195347628156, 0.8303721053466643, 1.0581474159307722, 0.7795295945044215, 0.3308276468591592, 0.9519032299358294, 1.1030335677176402, 0.7674287004159897]
{'CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(=N3)C)N4CCN(CC4)CCO', 'CC1=C(C=C(C=C1)C(=O)NC2=CC(=C(C=C2)CN3CCN(CC3)C)C(F)(F)F)C#CC4=CN=C5N4N=CC=C5', 'C1CN(C[C@@H]1O)C2=C(C=C(C=N2)C(=O)NC3=CC=C(C=C3)OC(F)(F)Cl)C4=CC=NN4', 'CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C4=CC=CC=N4', 'CC1=C(C=C(C=C1)C(=O)NC2=CC(=CC(=C2)C(F)(F)F)N3C=C(N=C3)C)NC4=NC=CC(=N4)C5=CN=CC=C5', 'CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5', 'CC1=C(C=C(C=C1)NC(=O)C2=CC(=C(C=C2)CN3CC[C@@H](C3)N(C)C)C(F)(F)F)NC4=NC=CC(=N4)C5=CN=CN=C5', 'CN1CCN(CC1)CCCOC2=C(C=C3C(=C2)N=CC(=C3NC4=CC(=C(C=C4Cl)Cl)OC)C#N)OC'}


## Run the below AFTER all the ligands and their SMILES have been run

In [ ]:
!pip install admet_ai
!pip install torch
!pip install argparse

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


In [ ]:
import torch
from argparse import Namespace
from admet_ai import ADMETModel

<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::shared_ptr<RDKit::FilterHierarchyMatcher> already registered; second conversion method ignored.


In [ ]:
torch.serialization.add_safe_globals([Namespace])

model = ADMETModel()

admet_df = model.predict(smiles=ligand_smiles_list)

# Lipinski drug-likeness filter
def lipinski_ok(mol):
    return (Descriptors.MolWt(mol) < 500 and
            Crippen.MolLogP(mol)  < 5   and
            Lipinski.NumHDonors(mol)    <= 5 and
            Lipinski.NumHAcceptors(mol) <= 10)

lipinski_flags = [lipinski_ok(Chem.MolFromSmiles(smi)) for smi in ligand_smiles_list]
admet_df["lipinski_pass"] = lipinski_flags

# Simple ADMET filters
good_admet = (
    (admet_df["HIA_prob"]        > 0.8) &   # absorption
    (admet_df["BBB_permeable"]   < 0.5) &   # avoid CNS
    (admet_df["hERG_inhib_prob"] < 0.3) &   # cardiotox
    (admet_df["CYP3A4_inhib"]    < 0.5) &   # DDI risk
    (admet_df["lipinski_pass"])
)
admet_df["admet_pass"] = good_admet

# Combine with binding scores and sort
rank_df = (
    admet_df
        .assign(binding_score=binding_scores)
        .sort_values(["admet_pass", "binding_score"], ascending=[False, False])
        .reset_index(drop=True)
)

print("FINAL RANKED LIGANDS")
print(rank_df[["SMILES", "binding_score", "admet_pass"]].head(10))


Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained parameter "readout.4.bias".
Moving model to cuda
Loading pretrained parameter "encoder.encoder.0.cached_zero_vector".
Loading pretrained parameter "encoder.encoder.0.W_i.weight".
Loading pretrained parameter "encoder.encoder.0.W_h.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.weight".
Loading pretrained parameter "encoder.encoder.0.W_o.bias".
Loading pretrained parameter "readout.1.weight".
Loading pretrained parameter "readout.1.bias".
Loading pretrained parameter "readout.4.weight".
Loading pretrained p

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([_reconstruct])` or the `torch.serialization.safe_globals([_reconstruct])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by 